# Close All Tradier Sandbox Positions (Iron Condor Multileg)

Emergency script to close ALL open option positions across ALL three
Tradier sandbox accounts (User, Matt, Logan).

Positions are grouped into Iron Condors by expiration and contract count,
then closed using the same cascade fallback as the live scanner:

1. Try 4-leg multileg close (buy_to_close shorts, sell_to_close longs)
2. If rejected → try 2x2-leg spread closes (put spread + call spread)
3. If rejected → try 4 individual leg closes

Any legs that don't group into a clean IC are closed individually.

In [0]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────
# Change these before running:

EXECUTE = False          # Set to True to actually close positions (False = dry run)
ACCOUNT_FILTER = None    # Set to "User", "Matt", or "Logan" to filter (None = all)

# ── Credentials (same as scanner — auto-populated if env vars are set) ──
import os

def _set_if_missing(key: str, fallback: str) -> None:
    if not os.environ.get(key):
        os.environ[key] = fallback

_set_if_missing("TRADIER_SANDBOX_KEY_USER", "iPidGGnYrhzjp6vGBBQw8HyqF0xj")
_set_if_missing("TRADIER_SANDBOX_KEY_MATT", "AGoNTv6o6GKMKT8uc7ooVNOct0e0")
_set_if_missing("TRADIER_SANDBOX_KEY_LOGAN", "AcDucIMyjeNgFh60LWOb0F5fhXHh")
_set_if_missing("TRADIER_SANDBOX_ACCOUNT_ID_USER", "VA39284047")
_set_if_missing("TRADIER_SANDBOX_ACCOUNT_ID_MATT", "VA55391129")
_set_if_missing("TRADIER_SANDBOX_ACCOUNT_ID_LOGAN", "VA59240884")

In [0]:
import re
import time
import logging
import requests
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import List, Dict, Optional, Tuple
from collections import defaultdict

CT = ZoneInfo("America/Chicago")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("close_all")

SANDBOX_URL = "https://sandbox.tradier.com/v1"

# OCC symbol regex: SPY260312P00580000
OCC_RE = re.compile(r"^([A-Z]{1,6})(\d{6})([PC])(\d{8})$")

In [ ]:
# ── OCC parsing ────────────────────────────────────────────────────────────

def parse_occ(symbol: str) -> Optional[Dict]:
    """Parse an OCC option symbol into its components."""
    m = OCC_RE.match(symbol)
    if not m:
        return None
    underlying, date_str, opt_type, strike_str = m.groups()
    strike = int(strike_str) / 1000.0
    expiration = f"20{date_str[:2]}-{date_str[2:4]}-{date_str[4:6]}"
    return {
        "underlying": underlying,
        "expiration": expiration,
        "type": opt_type,
        "strike": strike,
        "occ": symbol,
    }


# ── Tradier API helpers ────────────────────────────────────────────────────

def api_get(base_url: str, api_key: str, endpoint: str, params: Dict = None) -> Optional[Dict]:
    headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/json"}
    try:
        resp = requests.get(f"{base_url}{endpoint}", headers=headers, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        log.error(f"GET {endpoint} failed: {e}")
        return None


def api_post(base_url: str, api_key: str, endpoint: str, data: Dict = None) -> Optional[Dict]:
    headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/json"}
    try:
        resp = requests.post(f"{base_url}{endpoint}", headers=headers, data=data, timeout=15)
        # Capture body BEFORE raise_for_status so we always have it
        body = resp.text
        if resp.status_code >= 400:
            # Try to parse JSON error details
            try:
                error_json = resp.json()
                error_detail = error_json.get("errors", {}).get("error", [])
                if isinstance(error_detail, list):
                    error_detail = "; ".join(error_detail)
                elif isinstance(error_detail, str):
                    pass
                else:
                    error_detail = str(error_json)
            except Exception:
                error_detail = body[:500] if body else "(empty response)"
            log.error(f"POST {endpoint} [{resp.status_code}]: {error_detail}")
            return None
        return resp.json()
    except requests.exceptions.ConnectionError as e:
        log.error(f"POST {endpoint} connection error: {e}")
        return None
    except Exception as e:
        log.error(f"POST {endpoint} failed: {e}")
        return None


def get_positions(base_url: str, api_key: str, account_id: str) -> List[Dict]:
    """Fetch all open positions from a Tradier account."""
    data = api_get(base_url, api_key, f"/accounts/{account_id}/positions")
    if not data:
        return []
    positions = data.get("positions", {})
    if positions == "null" or not positions:
        return []
    pos_list = positions.get("position", [])
    if isinstance(pos_list, dict):
        return [pos_list]
    return pos_list if pos_list else []


def get_balance(base_url: str, api_key: str, account_id: str) -> Optional[Dict]:
    data = api_get(base_url, api_key, f"/accounts/{account_id}/balances")
    return data.get("balances", {}) if data else None

In [ ]:
# ── Iron Condor grouping ──────────────────────────────────────────────────

def group_into_iron_condors(positions: List[Dict]) -> Tuple[List[Dict], List[Dict]]:
    """
    Group raw Tradier positions into Iron Condors.

    Tradier merges positions with the same OCC symbol, so if you opened
    two ICs at different times with the same call strikes but different
    put strikes, the calls merge (e.g. -5 + -44 = -49) while the puts
    stay separate (-5 and -44).

    Strategy:
    1. Parse all positions into legs
    2. Match SPREAD PAIRS first: short+long on same type (P or C) with
       matching abs(qty) on the same underlying+expiration
    3. Combine a put spread + call spread into an IC when quantities match

    Returns: (iron_condors, ungrouped_legs)
    """
    parsed = []
    non_options = []
    for pos in positions:
        symbol = pos.get("symbol", "")
        qty = float(pos.get("quantity", 0))
        info = parse_occ(symbol)
        if info:
            info["qty"] = qty
            info["raw"] = pos
            parsed.append(info)
        else:
            non_options.append(pos)

    # Group by (underlying, expiration)
    by_exp = defaultdict(list)
    for p in parsed:
        key = (p["underlying"], p["expiration"])
        by_exp[key].append(p)

    iron_condors = []
    used = set()

    for (underlying, expiration), legs in by_exp.items():
        # Separate by type and direction
        short_puts = sorted([l for l in legs if l["type"] == "P" and l["qty"] < 0], key=lambda x: x["strike"])
        long_puts = sorted([l for l in legs if l["type"] == "P" and l["qty"] > 0], key=lambda x: x["strike"])
        short_calls = sorted([l for l in legs if l["type"] == "C" and l["qty"] < 0], key=lambda x: x["strike"])
        long_calls = sorted([l for l in legs if l["type"] == "C" and l["qty"] > 0], key=lambda x: x["strike"])

        # Step 1: Match put spreads (long put below short put, same abs qty)
        put_spreads = []
        used_sp = set()
        used_lp = set()
        for sp in short_puts:
            for lp in long_puts:
                if id(lp) in used_lp or id(sp) in used_sp:
                    continue
                if int(abs(sp["qty"])) == int(abs(lp["qty"])) and lp["strike"] < sp["strike"]:
                    put_spreads.append({"short": sp, "long": lp, "qty": int(abs(sp["qty"]))})
                    used_sp.add(id(sp))
                    used_lp.add(id(lp))

        # Step 2: Match call spreads (short call below long call, same abs qty)
        call_spreads = []
        used_sc = set()
        used_lc = set()
        for sc in short_calls:
            for lc in long_calls:
                if id(lc) in used_lc or id(sc) in used_sc:
                    continue
                if int(abs(sc["qty"])) == int(abs(lc["qty"])) and sc["strike"] < lc["strike"]:
                    call_spreads.append({"short": sc, "long": lc, "qty": int(abs(sc["qty"]))})
                    used_sc.add(id(sc))
                    used_lc.add(id(lc))

        # Step 3: Combine put spread + call spread into IC when qty matches
        used_ps = set()
        used_cs = set()
        for ps in put_spreads:
            for cs in call_spreads:
                if id(cs) in used_cs or id(ps) in used_ps:
                    continue
                if ps["qty"] == cs["qty"] and ps["short"]["strike"] < cs["short"]["strike"]:
                    ic = {
                        "underlying": underlying,
                        "expiration": expiration,
                        "contracts": ps["qty"],
                        "put_long": ps["long"]["strike"],
                        "put_short": ps["short"]["strike"],
                        "call_short": cs["short"]["strike"],
                        "call_long": cs["long"]["strike"],
                        "put_long_occ": ps["long"]["occ"],
                        "put_short_occ": ps["short"]["occ"],
                        "call_short_occ": cs["short"]["occ"],
                        "call_long_occ": cs["long"]["occ"],
                    }
                    iron_condors.append(ic)
                    used.update([
                        id(ps["short"]["raw"]), id(ps["long"]["raw"]),
                        id(cs["short"]["raw"]), id(cs["long"]["raw"]),
                    ])
                    used_ps.add(id(ps))
                    used_cs.add(id(cs))

        # Remaining unmatched spreads — combine with relaxed qty match
        for ps in put_spreads:
            if id(ps) not in used_ps:
                for cs in call_spreads:
                    if id(cs) not in used_cs:
                        min_qty = min(ps["qty"], cs["qty"])
                        if ps["short"]["strike"] < cs["short"]["strike"]:
                            ic = {
                                "underlying": underlying,
                                "expiration": expiration,
                                "contracts": min_qty,
                                "put_long": ps["long"]["strike"],
                                "put_short": ps["short"]["strike"],
                                "call_short": cs["short"]["strike"],
                                "call_long": cs["long"]["strike"],
                                "put_long_occ": ps["long"]["occ"],
                                "put_short_occ": ps["short"]["occ"],
                                "call_short_occ": cs["short"]["occ"],
                                "call_long_occ": cs["long"]["occ"],
                                "_note": f"qty mismatch: put={ps['qty']} call={cs['qty']}, using min={min_qty}",
                            }
                            iron_condors.append(ic)
                            used.update([
                                id(ps["short"]["raw"]), id(ps["long"]["raw"]),
                                id(cs["short"]["raw"]), id(cs["long"]["raw"]),
                            ])
                            used_cs.add(id(cs))
                            used_ps.add(id(ps))
                            break

    ungrouped = []
    for p in parsed:
        if id(p["raw"]) not in used:
            ungrouped.append(p["raw"])
    ungrouped.extend(non_options)

    return iron_condors, ungrouped


# ── Quick test with User account positions ────────────────────────────────
print("Testing IC grouping with User account positions...")
acct = get_all_accounts("User")[0]
positions = get_positions(SANDBOX_URL, acct["api_key"], acct["account_id"])
print(f"  Raw positions: {len(positions)}")
for p in positions:
    sym = p.get("symbol", "?")
    qty = float(p.get("quantity", 0))
    info = parse_occ(sym)
    exp = info["expiration"] if info else "?"
    strike = info["strike"] if info else "?"
    opt_type = info["type"] if info else "?"
    print(f"    {sym:30s}  qty={qty:>7.0f}  exp={exp}  strike={strike}  type={opt_type}")

ics, ungrouped = group_into_iron_condors(positions)
print(f"\n  Grouped into {len(ics)} Iron Condor(s):")
for i, ic in enumerate(ics, 1):
    note = ic.get("_note", "")
    suffix = f"  [!] {note}" if note else ""
    print(
        f"    IC #{i}: {ic['underlying']} {ic['expiration']}  "
        f"{ic['put_long']}/{ic['put_short']}P -- "
        f"{ic['call_short']}/{ic['call_long']}C  "
        f"x{ic['contracts']}{suffix}"
    )

print(f"\n  Ungrouped: {len(ungrouped)} leg(s)")
for p in ungrouped:
    sym = p.get("symbol", "?")
    qty = float(p.get("quantity", 0))
    print(f"    {sym:30s}  qty={qty:>7.0f}")

In [0]:
from typing import Dict, Optional

# ── Close IC with cascade fallback (4-leg -> 2x2-leg -> individual) ───────

def close_ic_multileg(
    base_url: str,
    api_key: str,
    account_id: str,
    ic: Dict,
    tag: str = "",
) -> Dict:
    """Close an Iron Condor using the same cascade as the live scanner."""
    contracts = str(ic["contracts"])
    ticker = ic["underlying"]

    # ── Attempt 1: 4-leg multileg close ──
    order_data = {
        "class": "multileg",
        "symbol": ticker,
        "type": "market",
        "duration": "day",
        "option_symbol[0]": ic["put_short_occ"],
        "side[0]": "buy_to_close",
        "quantity[0]": contracts,
        "option_symbol[1]": ic["put_long_occ"],
        "side[1]": "sell_to_close",
        "quantity[1]": contracts,
        "option_symbol[2]": ic["call_short_occ"],
        "side[2]": "buy_to_close",
        "quantity[2]": contracts,
        "option_symbol[3]": ic["call_long_occ"],
        "side[3]": "sell_to_close",
        "quantity[3]": contracts,
    }
    if tag:
        order_data["tag"] = tag[:255]

    result = api_post(base_url, api_key, f"/accounts/{account_id}/orders", data=order_data)
    if result:
        order = result.get("order", {})
        oid = order.get("id")
        status = order.get("status", "unknown")
        if oid and status not in ("rejected", "error"):
            return {"method": "4-leg", "success": True, "details": f"order_id={oid} [{status}]"}
        log.warning(f"4-leg rejected ({status}), trying 2x2-leg")

    # ── Attempt 2: 2x2-leg spread closes ──
    put_data = {
        "class": "multileg",
        "symbol": ticker,
        "type": "market",
        "duration": "day",
        "option_symbol[0]": ic["put_short_occ"],
        "side[0]": "buy_to_close",
        "quantity[0]": contracts,
        "option_symbol[1]": ic["put_long_occ"],
        "side[1]": "sell_to_close",
        "quantity[1]": contracts,
    }
    if tag:
        put_data["tag"] = f"{tag}-PUT"[:255]

    call_data = {
        "class": "multileg",
        "symbol": ticker,
        "type": "market",
        "duration": "day",
        "option_symbol[0]": ic["call_short_occ"],
        "side[0]": "buy_to_close",
        "quantity[0]": contracts,
        "option_symbol[1]": ic["call_long_occ"],
        "side[1]": "sell_to_close",
        "quantity[1]": contracts,
    }
    if tag:
        call_data["tag"] = f"{tag}-CALL"[:255]

    put_result = api_post(base_url, api_key, f"/accounts/{account_id}/orders", data=put_data)
    call_result = api_post(base_url, api_key, f"/accounts/{account_id}/orders", data=call_data)

    put_ok = False
    call_ok = False
    if put_result:
        o = put_result.get("order", {})
        put_ok = o.get("id") and o.get("status") not in ("rejected", "error")
    if call_result:
        o = call_result.get("order", {})
        call_ok = o.get("id") and o.get("status") not in ("rejected", "error")

    if put_ok and call_ok:
        pid = put_result["order"]["id"]
        cid = call_result["order"]["id"]
        return {"method": "2x2-leg", "success": True, "details": f"put_order={pid} call_order={cid}"}

    if put_ok or call_ok:
        log.warning(f"2x2-leg partial (put={'OK' if put_ok else 'FAIL'} call={'OK' if call_ok else 'FAIL'}), closing remaining legs individually")

    # ── Attempt 3: individual leg closes ──
    leg_results = []
    legs = [
        (ic["put_short_occ"], "buy_to_close", "PS"),
        (ic["put_long_occ"], "sell_to_close", "PL"),
        (ic["call_short_occ"], "buy_to_close", "CS"),
        (ic["call_long_occ"], "sell_to_close", "CL"),
    ]

    if put_ok:
        legs = [l for l in legs if l[2] not in ("PS", "PL")]
    if call_ok:
        legs = [l for l in legs if l[2] not in ("CS", "CL")]

    for occ, side, label in legs:
        leg_data = {
            "class": "option",
            "symbol": ticker,
            "option_symbol": occ,
            "side": side,
            "quantity": contracts,
            "type": "market",
            "duration": "day",
        }
        if tag:
            leg_data["tag"] = f"{tag}-{label}"[:255]

        leg_result = api_post(base_url, api_key, f"/accounts/{account_id}/orders", data=leg_data)
        if leg_result:
            o = leg_result.get("order", {})
            if o.get("id") and o.get("status") not in ("rejected", "error"):
                leg_results.append(f"{label}={o['id']}")
            else:
                leg_results.append(f"{label}=REJECTED")
        else:
            leg_results.append(f"{label}=FAILED")
        time.sleep(0.2)

    all_ok = all("REJECTED" not in r and "FAILED" not in r for r in leg_results)
    partial = any("REJECTED" not in r and "FAILED" not in r for r in leg_results)

    method_parts = []
    if put_ok:
        method_parts.append("put_spread")
    if call_ok:
        method_parts.append("call_spread")
    if leg_results:
        method_parts.append("individual")
    method = "+".join(method_parts) if method_parts else "individual"

    if all_ok or (put_ok and call_ok):
        return {"method": method, "success": True, "details": " ".join(leg_results)}
    elif partial or put_ok or call_ok:
        return {"method": method, "success": True, "details": f"PARTIAL: {' '.join(leg_results)}"}
    else:
        return {"method": "all_failed", "success": False, "details": " ".join(leg_results)}


def close_single_leg(
    base_url: str,
    api_key: str,
    account_id: str,
    pos: Dict,
    tag: str = "",
) -> Optional[Dict]:
    """Close a single ungrouped position (option or equity)."""
    symbol = pos.get("symbol", "")
    qty = float(pos.get("quantity", 0))
    is_option = OCC_RE.match(symbol) is not None

    if is_option:
        info = parse_occ(symbol)
        underlying = info["underlying"] if info else symbol[:3]
        side = "sell_to_close" if qty > 0 else "buy_to_close"
        order_data = {
            "class": "option",
            "symbol": underlying,
            "option_symbol": symbol,
            "side": side,
            "quantity": str(int(abs(qty))),
            "type": "market",
            "duration": "day",
        }
    else:
        side = "sell" if qty > 0 else "buy_to_cover"
        order_data = {
            "class": "equity",
            "symbol": symbol,
            "side": side,
            "quantity": str(int(abs(qty))),
            "type": "market",
            "duration": "day",
        }

    if tag:
        order_data["tag"] = tag[:255]

    result = api_post(base_url, api_key, f"/accounts/{account_id}/orders", data=order_data)
    if result:
        order = result.get("order", {})
        return {"order_id": order.get("id"), "status": order.get("status", "unknown")}
    return None

In [0]:
# ── Account discovery ─────────────────────────────────────────────────────

def get_all_accounts(account_filter: str = None) -> List[Dict]:
    accounts = []
    for label, env_key, env_id in [
        ("User", "TRADIER_SANDBOX_KEY_USER", "TRADIER_SANDBOX_ACCOUNT_ID_USER"),
        ("Matt", "TRADIER_SANDBOX_KEY_MATT", "TRADIER_SANDBOX_ACCOUNT_ID_MATT"),
        ("Logan", "TRADIER_SANDBOX_KEY_LOGAN", "TRADIER_SANDBOX_ACCOUNT_ID_LOGAN"),
    ]:
        key = os.environ.get(env_key, "")
        acct_id = os.environ.get(env_id, "")
        if key and acct_id:
            accounts.append({
                "name": label,
                "api_key": key,
                "account_id": acct_id,
                "base_url": SANDBOX_URL,
            })

    if account_filter:
        accounts = [a for a in accounts if a["name"].lower() == account_filter.lower()]

    return accounts

In [0]:
# ── MAIN ──────────────────────────────────────────────────────────────────

now = datetime.now(CT)
mode = "EXECUTE" if EXECUTE else "DRY RUN"

print("=" * 70)
print(f"  Close All Tradier Positions (IC Multileg) — {mode}")
print(f"  Time: {now.strftime('%Y-%m-%d %H:%M:%S CT')}")
print(f"  Cascade: 4-leg -> 2x2-leg -> individual legs")
print("=" * 70)

accounts = get_all_accounts(ACCOUNT_FILTER)
if not accounts:
    print("\n  No Tradier accounts found. Check environment variables.")
else:
    print(f"\n  Found {len(accounts)} account(s): {', '.join(a['name'] for a in accounts)}")

total_ics = 0
total_ungrouped = 0
total_closed = 0
total_failed = 0

for acct in accounts:
    name = acct["name"]
    base_url = acct["base_url"]
    api_key = acct["api_key"]
    account_id = acct["account_id"]
    is_sandbox = "sandbox" in base_url

    print(f"\n{'─' * 70}")
    print(f"  Account: {name} ({'SANDBOX' if is_sandbox else 'PRODUCTION'}) — {account_id}")
    print(f"{'─' * 70}")

    balance = get_balance(base_url, api_key, account_id)
    if balance:
        equity = balance.get("total_equity", balance.get("equity", "?"))
        print(f"  Equity: ${equity}")

    positions = get_positions(base_url, api_key, account_id)
    if not positions:
        print("  No open positions.")
        continue

    print(f"  {len(positions)} raw position leg(s)")

    ics, ungrouped = group_into_iron_condors(positions)
    total_ics += len(ics)
    total_ungrouped += len(ungrouped)

    if ics:
        print(f"\n  Iron Condors ({len(ics)}):")
        for i, ic in enumerate(ics, 1):
            print(
                f"    IC #{i}: {ic['underlying']} {ic['expiration']}  "
                f"{ic['put_long']}/{ic['put_short']}P — "
                f"{ic['call_short']}/{ic['call_long']}C  "
                f"x{ic['contracts']}"
            )

            if not EXECUTE:
                print(f"           [DRY RUN] Would close as 4-leg multileg @ market")
                continue

            tag = f"CLOSE_ALL_{now.strftime('%Y%m%d_%H%M')}"
            result = close_ic_multileg(base_url, api_key, account_id, ic, tag)

            if result["success"]:
                total_closed += 1
                print(f"           CLOSED [{result['method']}]: {result['details']}")
            else:
                total_failed += 1
                print(f"           FAILED [{result['method']}]: {result['details']}")

            time.sleep(0.5)

    if ungrouped:
        print(f"\n  Ungrouped legs ({len(ungrouped)}):")
        for pos in ungrouped:
            symbol = pos.get("symbol", "???")
            qty = float(pos.get("quantity", 0))
            side_label = "BUY_TO_CLOSE" if qty < 0 else "SELL_TO_CLOSE"
            print(f"    {symbol:30s}  qty={qty:>6.0f}  → {side_label}")

            if not EXECUTE:
                print(f"      [DRY RUN] Would close individually @ market")
                continue

            tag = f"CLOSE_ALL_{now.strftime('%Y%m%d_%H%M')}"
            result = close_single_leg(base_url, api_key, account_id, pos, tag)
            if result and result.get("order_id"):
                total_closed += 1
                print(f"      CLOSED: order_id={result['order_id']} [{result['status']}]")
            else:
                total_failed += 1
                print(f"      FAILED to close!")

            time.sleep(0.3)

# Summary
print(f"\n{'=' * 70}")
print(f"  SUMMARY")
print(f"  Iron Condors found:    {total_ics}")
print(f"  Ungrouped legs found:  {total_ungrouped}")
if EXECUTE:
    print(f"  Successfully closed:   {total_closed}")
    print(f"  Failed to close:       {total_failed}")
else:
    print(f"  [DRY RUN] — no orders sent. Set EXECUTE = True and re-run.")
print(f"{'=' * 70}")